# Real precipitation — styled plot **and animation**

A real **MSWEP daily precipitation** series (mm/day) over the Rhine basin, read frame-by-frame with pyramids
(GDAL) and rendered with cleopatra's `total_precipitation` style. A fixed 0…max scale keeps wet and dry days
comparable across the animation. A second animation shows the matching **river discharge** (`Qtot`) series.

In [ ]:
%matplotlib inline
import os, sys, glob
_wt = r"C:/python-environments/worktrees/cleopatra/perceptual-palettes/src"
if os.path.isdir(_wt) and _wt not in sys.path:
    sys.path.insert(0, _wt)
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import Normalize
from pyramids.dataset import Dataset

import cleopatra
from cleopatra.array_glyph import ArrayGlyph, FrameLabel
from cleopatra.colors import apply_data_style, resolve_single_layer_style
from cleopatra.animation import embed_gif
print("cleopatra", cleopatra.__version__)

PYR = Path(r"C:/gdrive/algorithms/gis/pyramids/examples/data")

def read_stack(pattern):
    """Read a folder of dated rasters into (3D stack, extent, date-labels)."""
    files = sorted(glob.glob(str(pattern)))
    frames, labels = [], []
    ext = None
    for f in files:
        ds = Dataset.read_file(f)
        a = np.asarray(ds.read_array(), dtype=float)
        if a.ndim == 3: a = a[0]
        nod = ds.no_data_value[0] if ds.no_data_value else None
        if nod is not None and np.isfinite(nod): a = np.where(np.isclose(a, nod), np.nan, a)
        frames.append(a)
        r, c = a.shape; x0, dx, _, y0, _, dy = ds.geotransform
        ext = [x0, x0 + c * dx, y0 + r * dy, y0]
        stem = Path(f).stem            # e.g. MSWEP_1979.01.01 / Qtot_1979-01-01
        labels.append(stem.split("_")[-1].replace(".", "-"))
    return np.stack(frames), ext, labels

precip, p_ext, p_labels = read_stack(PYR / "geotiff/raster-folder/MSWEP_*.tif")
print("MSWEP precip stack:", precip.shape, "|", p_labels[0], "->", p_labels[-1],
      "| max", round(float(np.nanmax(precip)), 1), "mm/day")

## The wettest day

One frame with the `total_precipitation` style and a colorbar.

In [ ]:
pmax = float(np.nanmax(precip))
day = int(np.nanargmax([np.nansum(f) for f in precip]))
_, pcfg = resolve_single_layer_style("total_precipitation")

fig, ax = plt.subplots(figsize=(6.5, 7))
apply_data_style(ax, {"total_precipitation": precip[day]}, style="total_precipitation",
                 vmin=0, vmax=pmax, extent=p_ext, origin="upper", legend=False)
sm = plt.cm.ScalarMappable(norm=Normalize(0, pmax), cmap=pcfg["cmap"])
fig.colorbar(sm, ax=ax, fraction=0.045, pad=0.02, label="precipitation (mm/day)")
ax.set_title(f"MSWEP precipitation — {p_labels[day]} (wettest day)")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

## Animated — the daily precipitation series

Each frame is the same `total_precipitation` style on that day, on a fixed 0…max scale.

In [ ]:
# figure sized to the data box so the equal-aspect map fills it (no letterbox)
pw, pe, ps, pn = p_ext
fw, fh = pe - pw, pn - ps
fs = (8, 8 * fh / fw) if fw >= fh else (8 * fw / fh, 8)
glyph = ArrayGlyph(precip, extent=[pw, ps, pe, pn], figsize=fs)
anim = glyph.animate(p_labels, style="total_precipitation", vmin=0, vmax=pmax,
                     title="MSWEP daily precipitation",
                     frame_label=FrameLabel(location=[pw + 0.03 * (pe - pw),
                                                      ps + 0.05 * (pn - ps)]),
                     interval=500)
plt.close(glyph.fig)
embed_gif(anim, fps=2)

## Bonus — river discharge (`Qtot`) series

The matching Rhine runoff series (10 days), rendered with the `flow_accumulation` style (symmetric-log,
value-linked opacity) so the wet channel network stands out.

In [ ]:
q, q_ext, q_labels = read_stack(PYR / "geotiff/rhine/Qtot_*.tif")
qmax = float(np.nanmax(q))
qw, qe, qs, qn = q_ext
fw, fh = qe - qw, qn - qs
fs = (8, 8 * fh / fw) if fw >= fh else (8 * fw / fh, 8)
glyph = ArrayGlyph(q, extent=[qw, qs, qe, qn], figsize=fs)
anim2 = glyph.animate(q_labels, style="flow_accumulation", vmin=0, vmax=qmax,
                      title="Rhine discharge Qtot",
                      frame_label=FrameLabel(location=[qw + 0.03 * (qe - qw),
                                                       qs + 0.05 * (qn - qs)]),
                      interval=400)
plt.close(glyph.fig)
embed_gif(anim2, fps=3)